# Slide Exercise 02: SBERT Semantic Movie Recommender

This is the refined version of `SBERT_MovieRecommender.ipynb`.

Learning objectives:
- Compare lexical TF-IDF similarity with semantic embedding similarity.
- Use `all-MiniLM-L6-v2` when available.
- Keep the exercise runnable with a TF-IDF fallback.

Main functions used:
- `SentenceTransformer(...)`: loads a pretrained sentence embedding model.
- `model.encode(...)`: converts descriptions into dense semantic vectors.
- `cosine_similarity(...)`: compares embedding vectors.
- `try/except`: keeps optional model code from breaking the notebook.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


,movie_id,title,genres,director,year,duration_min,rating,family_friendly,description,keywords
0,1,Inception,Sci-Fi|Thriller|Action,Christopher Nolan,2010,148,8.8,0,A thief enters layered dreams to plant an idea...,dreams heist subconscious mind-bending
1,2,Interstellar,Sci-Fi|Adventure|Drama,Christopher Nolan,2014,169,8.7,0,Astronauts travel through a wormhole to find a...,space exploration wormhole survival family
2,3,Titanic,Romance|Drama,James Cameron,1997,195,7.9,0,A young couple from different social classes f...,romance ship tragedy historical
3,4,The Matrix,Sci-Fi|Action,The Wachowskis,1999,136,8.7,0,A hacker discovers that reality is a simulated...,simulation hacker reality action cyberpunk
4,5,Toy Story,Animation|Adventure|Comedy|Family,John Lasseter,1995,81,8.3,1,A cowboy doll feels threatened when a space ra...,toys friendship family adventure


Build a text field that reads like a short movie profile.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

movies["profile_text"] = movies["title"] + ". " + movies["description"] + " Genres: " + movies["genres"].str.replace("|", ", ", regex=False)

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["profile_text"])
tfidf_similarity = cosine_similarity(tfidf_matrix)


Try SBERT. If it is missing or cannot load the model, use the TF-IDF matrix instead.


In [3]:
model_name = "TF-IDF fallback"
semantic_similarity = tfidf_similarity
semantic_vectors = tfidf_matrix

try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    semantic_vectors = model.encode(movies["profile_text"].tolist(), show_progress_bar=False)
    semantic_similarity = cosine_similarity(semantic_vectors)
    model_name = "SBERT all-MiniLM-L6-v2"
except Exception as exc:
    print("SBERT is optional for this exercise. Continuing with TF-IDF fallback.")
    print(type(exc).__name__, str(exc)[:160])

model_name


/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google 

'SBERT all-MiniLM-L6-v2'

Use one ranking function for both lexical and semantic similarities.


In [4]:
def recommend(title, similarity_matrix, label, n=5):
    idx = movies.index[movies["title"].eq(title)][0]
    ranked = similarity_matrix[idx].argsort()[::-1]
    rows = []
    for other_idx in ranked:
        if other_idx == idx:
            continue
        rows.append({
            "method": label,
            "input_movie": title,
            "recommended_movie": movies.loc[other_idx, "title"],
            "score": round(float(similarity_matrix[idx, other_idx]), 3),
        })
        if len(rows) == n:
            break
    return pd.DataFrame(rows)

pd.concat([
    recommend("Gravity", tfidf_similarity, "TF-IDF"),
    recommend("Gravity", semantic_similarity, model_name),
], ignore_index=True)


,method,input_movie,recommended_movie,score
0,TF-IDF,Gravity,Interstellar,0.239
1,TF-IDF,Gravity,The Martian,0.174
2,TF-IDF,Gravity,Inception,0.169
3,TF-IDF,Gravity,The Matrix,0.099
4,TF-IDF,Gravity,The Notebook,0.047
5,SBERT all-MiniLM-L6-v2,Gravity,Interstellar,0.683
6,SBERT all-MiniLM-L6-v2,Gravity,The Martian,0.602
7,SBERT all-MiniLM-L6-v2,Gravity,Titanic,0.575
8,SBERT all-MiniLM-L6-v2,Gravity,Toy Story,0.466
9,SBERT all-MiniLM-L6-v2,Gravity,Inception,0.428


Interpretation:

TF-IDF rewards shared words. SBERT, when available, can connect descriptions that express similar meaning with different words.

Student task:
1. Compare recommendations for `Titanic`.
2. Add a new movie with a description that uses different words for a similar idea.
